<a href="https://colab.research.google.com/github/linda-r-wu/Problem-Set-1-Econ_5200_PS1/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [2]:
# Clean consumer transactions
rng = np.random.default_rng(42)

basket_values = rng.lognormal(mean=3.5, sigma=0.7, size=10000)
print(f"Median consumer basket: ${np.median(basket_values):.2f}")

consumer = pd.DataFrame({
    "customer_id": list(range(10000)) + list(range(10000)),
    "year": [2024] * 10000 + [2025] * 10000,
    "basket_value": list(basket_values) + list(basket_values),
    "customer_type": "consumer",
    "status": "completed"
})

print(consumer.shape)

Median consumer basket: $32.81
(20000, 5)


In [3]:
# The truth: the consumer average per year, recorded before contamination
true_mean = consumer.groupby("year")["basket_value"].mean()

print("True consumer averages:")
print(true_mean.round(2))

df = consumer.copy()

True consumer averages:
year
2024    42.15
2025    42.15
Name: basket_value, dtype: float64


In [4]:
# Contamination 1: B2B orders: more large B2B orders appear in the second year.
b2b = pd.DataFrame({
    "customer_id": range(20000, 20030),
    "year": [2024] * 10 + [2025] * 20,
    "basket_value": [1000.0] * 30,
    "customer_type": "B2B",
    "status": "completed"
})

df = pd.concat([df, b2b], ignore_index=True)

In [5]:
# Contamination 2: what gets logged changes between year 1 and year 2: cancelled orders are logged only in 2025
cancelled = pd.DataFrame({
    "customer_id": range(30000, 30200),
    "year": [2025] * 200,
    "basket_value": [180.0] * 200,
    "customer_type": "consumer",
    "status": "cancelled"
})

df = pd.concat([df, cancelled], ignore_index=True)

print("Contaminated dataset:")
print(df.groupby(["year", "customer_type", "status"]).size())

Contaminated dataset:
year  customer_type  status   
2024  B2B            completed       10
      consumer       completed    10000
2025  B2B            completed       20
      consumer       cancelled      200
                     completed    10000
dtype: int64


notes: i used identical consumer basket values in both years so that the consumer average has zero growth. then, i added more of the larger b2b orders in the second year and added cancelled orders even though neither should be in the completed consumer basket metric.

### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [6]:
# The dashboard's number vs the truth
# first average the logged baskets
naive_mean = df.groupby("year")["basket_value"].mean()

comparison = pd.DataFrame({
    "True consumer mean": true_mean,
    "Dashboard mean": naive_mean
})

# next find the dollar gap for each year
comparison["Gap ($)"] = naive_mean - true_mean
print(comparison.round(2))

# and compare it with growth rates
true_growth = (true_mean.loc[2025] / true_mean.loc[2024] - 1) * 100
naive_growth = (naive_mean.loc[2025] / naive_mean.loc[2024] - 1) * 100

print(f"\nTrue growth: {true_growth:.2f}%")
print(f"Dashboard growth: {naive_growth:.2f}%")
print(f"Growth gap: {naive_growth - true_growth:.2f} percentage points")

      True consumer mean  Dashboard mean  Gap ($)
year                                             
2024               42.15           43.11     0.96
2025               42.15           46.72     4.57

True growth: 0.00%
Dashboard growth: 8.39%
Growth gap: 8.39 percentage points


notes: dashboard shows 8.39% increase in average basket value when true consumer average has 0% growth. this gap shows a change in transactions included instead of an increase in completed consumer basket values since we included the cancelled orders and b2b orders.

---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [7]:
# How much of the gap is the skew and the B2B tail?
# keeping completed numbers
completed = df[df["status"] == "completed"]
completed_mean = completed.groupby("year")["basket_value"].mean()

completed_growth = (
    completed_mean.loc[2025] / completed_mean.loc[2024] - 1
) * 100

# removing b2b from completed orders brings us back to consumer truth
b2b_contribution = completed_growth - true_growth

print(f"B2B contribution: {b2b_contribution:.2f} percentage points")

B2B contribution: 2.22 percentage points


In [8]:
# How much is the change in what was logged?
# comparing original dashboard with logging repaired dashboard
logging_contribution = naive_growth - completed_growth
total_gap = naive_growth - true_growth

print(f"Logging contribution: {logging_contribution:.2f} percentage points")
print(f"Total growth gap: {total_gap:.2f} percentage points")

assert np.isclose(
    b2b_contribution + logging_contribution,
    total_gap
)

Logging contribution: 6.17 percentage points
Total growth gap: 8.39 percentage points


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |2.22 pp|removed cancelled orders and compared growth to true consumer growth|
| Change in what was logged |6.17 pp|compared original dashbaord growth with the growth when cancelled orders are removed|

notes: the inclusion of b2b is causing the distortion in growth we are seeing after removing cancelled orders. the logging contribution is the change in reported growth after cancelled orders are removed, whcih add to the gap. we cant guarantee the right skew is an error alone, but the b2b orders are definitely creating a problem.

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [9]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
completed = df[df["status"] == "completed"]

alternatives = pd.DataFrame(index=[2024, 2025])

for year in [2024, 2025]:
    year_data = completed[completed["year"] == year]
    x = year_data["basket_value"]

    alternatives.loc[year, "Median"] = x.median()
    alternatives.loc[year, "Trimmed mean"] = stats.trim_mean(x, 0.1)

    # Exclude B2B using its recorded customer type, not basket size.
    consumer_only = year_data[year_data["customer_type"] == "consumer"]
    alternatives.loc[year, "Mean after B2B exclusion"] = (
        consumer_only["basket_value"].mean()
    )

print(alternatives.round(2))

      Median  Trimmed mean  Mean after B2B exclusion
2024   32.84         36.58                     42.15
2025   32.87         36.63                     42.15


| Estimator | Value 2024 | Value 2025 | The assumption that makes it right | When it would be wrong |
|---|---|---|---|---|
| Median |32.84|32.87| summarizing the typical completed basket|if we are looking at average revenue per consumer transaction|
| Trimmed mean |36.58|36.63|showing middle 80% of completed baskets with less influence from extremes|b2b orders that survive trimming, or large orders that are real revenue|
| Mean after B2B exclusion |42.15|42.15|revenue per completed consumer transaction|mistakenly including b2b, or customer types incorrectly classified|

**Which number should the dashboard show?** Commit to one and defend it.

the dashboard should show the mean after b2b exclusion ($42.15) because it shows 0% growth, and maintains large orders that are legitimate while still excluding unnecessary transactions. the dashboard needs the average revenue per completed consumer transaction, and this is the only one that would accurately represent this.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [10]:
import os
os.makedirs("src", exist_ok=True)

In [11]:
%%writefile src/basket_metrics.py
"""YOUR ONE-LINE DESCRIPTION OF THE MODULE."""
"""audit transaction data and summaries of basket values"""

import numpy as np
import pandas as pd


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """return missing percentages, dtype, skew and outlier count per column."""
    report = pd.DataFrame(index=df.columns)
    report["missing_pct"] = df.isna().mean() * 100
    report["dtype"] = df.dtypes.astype(str)
    report["skew"] = np.nan
    report["outlier_count"] = np.nan

    for col in df.select_dtypes(include="number").columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        outliers = (
            (df[col] < q1 - 1.5 * iqr) |
            (df[col] > q3 + 1.5 * iqr)
        )

        report.loc[col, "skew"] = df[col].skew()
        report.loc[col, "outlier_count"] = outliers.sum()

    return report


def robust_mean(x, method: str = "median") -> float:
    """Return median, 10%-per-tail trimmed mean, or completed consumer mean."""
    if method not in ["median", "trimmed", "rule"]:
        raise ValueError("method must be median, trimmed or rule")

    if method == "rule":
        x = x[
            (x["customer_type"] == "consumer") &
            (x["status"] == "completed")
        ]
        x = x["basket_value"]

    x = pd.Series(x).dropna()

    if len(x) == 0:
        raise ValueError("No values available")

    if method == "median":
        return float(x.median())

    elif method == "trimmed":
        x = x.sort_values()
        trim = int(len(x) * 0.1)
        return float(x.iloc[trim:len(x) - trim].mean())

    else:
        return float(x.mean())


if __name__ == "__main__":
    # A short demonstration that runs when the file is run as a script
    demo = pd.DataFrame({
        "basket_value": [20, 30, 40, 1000],
        "customer_type": ["consumer", "consumer", "consumer", "B2B"],
        "status": ["completed"] * 4
    })

    print(audit_report(demo))
    print("Median:", robust_mean(demo["basket_value"], "median"))
    print("Trimmed:", robust_mean(demo["basket_value"], "trimmed"))
    print("Consumer mean:", robust_mean(demo, "rule"))

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [12]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance

# test 1: confirming median is 20
test_values = pd.Series([10, 20, 30])
assert basket_metrics.robust_mean(test_values, "median") == 20

# test 2: trimming leaves only 10 removes 0's and 100's
test_trimmed = pd.Series([0, 10, 10, 10, 10, 10, 10, 10, 10, 100])
assert basket_metrics.robust_mean(test_trimmed, "trimmed") == 10

# test 3: removing cancelled and b2b orders
test_data = pd.DataFrame({
    "basket_value": [10, 20, 1000, 200],
    "customer_type": ["consumer", "consumer", "B2B", "consumer"],
    "status": ["completed", "completed", "completed", "cancelled"]
})
assert basket_metrics.robust_mean(test_data, "rule") == 15

print("All three tests passed.")

All three tests passed.


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`: https://github.com/linda-r-wu/Problem-Set-1-Econ_5200_PS1/blob/main/ai-appendix.md**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

In [13]:
# strongest objection: Why trim legitimate consumer purchases? Will this number reconcile with revenue?” A trimmed mean removes real spending from both ends and measures the middle of the distribution rather than average revenue per transaction.

# evidence: After excluding B2B and cancelled orders, compare the trimmed mean with total consumer revenue divided by completed consumer transactions. Quantify the difference.

# The analysis that settles it

corrected = df[
    (df["customer_type"] == "consumer") &
    (df["status"] == "completed")
]

for year in [2024, 2025]:
    x = corrected[corrected["year"] == year]["basket_value"]

    revenue_average = x.sum() / len(x)
    trimmed_average = stats.trim_mean(x, 0.1)
    difference = trimmed_average - revenue_average

    print(f"\n{year}")
    print(f"Revenue per transaction: ${revenue_average:.2f}")
    print(f"Trimmed consumer mean: ${trimmed_average:.2f}")
    print(f"Difference: ${difference:.2f}")


2024
Revenue per transaction: $42.15
Trimmed consumer mean: $36.53
Difference: $-5.62

2025
Revenue per transaction: $42.15
Trimmed consumer mean: $36.53
Difference: $-5.62


**What you found, including if it changes your Phase 2 recommendation:**

we found that after excluding b2b and cancelled orders, revenue per consumer was 42.15 in both years, while the trimmed mean was 36.53. this was 5.62 lower which means trimming removes some valid consumer spending even when we fix the contamination. this validates my phase 2 recommendation to use the untrimmed mean of completed consumer baskets, but remove the b2b and cancelled orders.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

#### Reported basket growth: 8.39%; corrected growth: 0%

Correction: By using the mean of completed consumer transactions, and excluding B2B and cancelled orders, the corrected average is 42.15 dollars in both years (2024-2025). When only excluding cancelled orders, B2B is contributing 2.22 percentage points, and logging change is contributing 6.17 percentage points to the reported growth. We use the mean that removes B2B and cancelled orders because trimming the mean would leave out valuable customer baskets that are legitimate spending and lowering the average to 36.53 dollars which is removing 5.62 dollars from the reported number.

Confidence: High- this simulation is controlled, and the corrected mean matches the truth that was recorded before the contamination portion.

Before Acting: We would double/cross check our consumer sales totals and completed order counts against the finance records; then, agree on how to handle refunds, discounts, and taxes.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors